
# Community Safety Synthetic Data — Generator (Notebook 1)

This notebook generates an **event-level synthetic dataset (2015–2025)** suitable for descriptive analytics and forecasting.  
It includes domain-inspired seasonality, trend, territory and zip effects, and optional **gentle dirty-data injections** for wrangling practice.

**Outputs** (saved in the same environment):
- `synthetic_sound_the_alarm_2015_2025.csv` (clean base)
- `synthetic_sound_the_alarm_2015_2025_dirty.csv` (optional, with nulls & typos)

> Tip: Run this notebook end-to-end to produce files, then use Notebook 2 for **Data Wrangling & Cleaning**.


In [1]:

import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import math

np.random.seed(42)  # reproducibility

# -----------------------------
# Config knobs (edit as needed)
# -----------------------------
START_DATE = pd.Timestamp("2015-01-01")
END_DATE   = pd.Timestamp("2025-09-20")

# Territories and relative activity
TERRITORIES = {
    "LA Metro":      {"zip_prefixes": ["900", "901"], "base_rate": 1.90},
    "Orange County": {"zip_prefixes": ["926", "927"], "base_rate": 1.20},
    "Inland Empire": {"zip_prefixes": ["923", "924", "925"], "base_rate": 1.35},
    "San Diego":     {"zip_prefixes": ["919", "920", "921"], "base_rate": 1.10},
    "Central Coast": {"zip_prefixes": ["931", "934"], "base_rate": 0.75},
}

AVG_EVENTS_PER_DAY_REGION = 6.0  # baseline

# Dirty-data injection toggles
INJECT_DIRTY = True
NULL_FRAC_FIRE_ESCAPE = 0.03       # ~3% NaNs in fire_escape_plans_num
TYPO_FRAC_TERRITORY    = 0.02       # ~2% territory typos
OUTLIER_FRAC_ALARMS    = 0.003      # ~0.3% big alarm counts added
KEEP_ZERO_EVENT_FRAC   = 0.10       # keep 10% of all-zero rows


In [2]:

# -----------------------------
# Helper functions
# -----------------------------

def covid_multiplier(date: pd.Timestamp) -> float:
    # Deep dip in 2020-05 to 2021-03; partial in 2021-04 to 2021-12
    if pd.Timestamp("2020-05-01") <= date <= pd.Timestamp("2021-03-31"):
        return 0.35
    if pd.Timestamp("2021-04-01") <= date <= pd.Timestamp("2021-12-31"):
        return 0.70
    return 1.0

def trend_multiplier(date: pd.Timestamp) -> float:
    years_since_2015 = (date.year + (date.dayofyear/365.25)) - 2015.0
    return 1.0 + 0.03 * years_since_2015  # ~3% per year growth

def month_multiplier(month: int) -> float:
    seasonal = {
        1: 0.95, 2: 0.95, 3: 1.05, 4: 1.20, 5: 1.25, 6: 1.15,
        7: 0.90, 8: 0.92, 9: 1.05, 10: 1.10, 11: 1.10, 12: 0.85
    }
    return seasonal.get(month, 1.0)

def dow_multiplier(dow: int) -> float:  # Monday=0
    v = {0: 0.95, 1: 1.00, 2: 1.05, 3: 1.05, 4: 1.00, 5: 0.85, 6: 1.30}
    return v.get(dow, 1.0)

def draw_zip(prefixes):
    pr = np.random.choice(prefixes)
    # Draw within plausible suffix ranges by prefix
    if pr in ["900", "901"]:
        suffix = np.random.randint(1, 99)   # 90001–90098 approx
    elif pr in ["926", "927", "923", "924", "925", "919", "920", "921", "931", "934"]:
        suffix = np.random.randint(0, 99)
    else:
        suffix = np.random.randint(0, 999)
    return f"{pr}{suffix:02d}"

_zip_effect_cache = {}
def get_zip_effect(zipcode: str) -> float:
    # Multiplicative, centered at 1 with mild variance
    if zipcode not in _zip_effect_cache:
        _zip_effect_cache[zipcode] = np.exp(np.random.normal(loc=0.0, scale=0.15))
    return _zip_effect_cache[zipcode]


## Generate Base (Clean) Synthetic Dataset

In [6]:

all_days = pd.date_range(START_DATE, END_DATE, freq="D")
rows = []

territory_names = list(TERRITORIES.keys())
base_rates = np.array([TERRITORIES[t]["base_rate"] for t in territory_names])

for d in all_days:
    lam_day = (
        AVG_EVENTS_PER_DAY_REGION
        * month_multiplier(d.month)
        * dow_multiplier(d.dayofweek)
        * trend_multiplier(d)
        * covid_multiplier(d)
    )
    lam_day = max(0.1, min(lam_day, 60))  # safety cap
    num_events = np.random.poisson(lam_day)  # lam_day = expected frequency, num_events = realized frequency.

    probs = base_rates / base_rates.sum()
    assigned_territories = np.random.choice(territory_names, size=num_events, p=probs) # gives you expected installs per day

    for tname in assigned_territories:
        tinfo = TERRITORIES[tname]
        zip_code = draw_zip(tinfo["zip_prefixes"])
        zip_eff  = get_zip_effect(zip_code)

        sat_boost = 1.15 if d.dayofweek == 6 else 1.0 # if Sundays (dow=6) get ~+15% activity
        # hh = household; hh_scale = household-level multiplier adding per-visit variation
        hh_scale  = np.clip(np.random.normal(loc=1.0 * sat_boost * zip_eff, scale=0.15), 0.6, 1.6)

        # sa = actual number of smoke alarms installed for this event.
        # First compute sa_mean (the expected installs, a float), then sample from a
        # negative binomial distribution to introduce realistic variability.
        # This produces an integer outcome (0, 1, 2, …), reflecting that some visits
        # install fewer alarms than average and some install more.

        base_sa = 3.2 * TERRITORIES[tname]["base_rate"] / base_rates.mean() 
        sa_mean = base_sa * month_multiplier(d.month) * 0.9 * hh_scale
        sa_mean = np.clip(sa_mean, 0.6, 8.0)
        sa = np.random.negative_binomial(n=2, p=2/(2+sa_mean))
        sa = int(max(0, sa))

        # ppl = actual number of people served in this event.
        # expected_people is estimated from smoke alarms installed (sa), scaled by household size,
        # random noise, and tuning factors. Then draw from a Poisson distribution to get an
        # integer count (0, 1, 2, …).

        expected_people = 1.6 * (sa ** 0.85) * 0.9 * (0.9 + 0.2*np.random.rand()) * hh_scale
        ppl = int(np.random.poisson(lam=max(0.2, expected_people)))

        # fep = number of fire escape plans created for this event.
        # fep_lambda sets the expected value (baseline 0.6, boosted on Saturdays, with random noise),
        # then Poisson draw converts it to an integer outcome.

        fep_lambda = 0.6 * sat_boost * (0.8 + 0.4*np.random.rand())
        fep = int(np.random.poisson(lam=fep_lambda))

        # haz = number of additional hazards found.
        # haz_base sets the expected value (baseline 0.4, adjusted by ZIP and random noise).
        # Poisson draw produces an integer, clipped to [0, 4] for realism.

        haz_base = 0.4 * zip_eff * (0.8 + 0.5*np.random.rand())
        haz = int(np.clip(np.random.poisson(lam=haz_base), 0, 4))

        rows.append({
            "date": d.date(),
            "zip": zip_code,
            "territory": tname,
            "people_served_num": ppl,
            "smoke_alarms_installed_num": sa,
            "fire_escape_plans_num": fep,
            "additional_hazards_num": haz,
        })

df = pd.DataFrame(rows)

# Keep 10% of all-zero rows; drop the rest; we'll know which rows are "all zeros"
mask_all_zero = (
    (df["people_served_num"] == 0) &
    (df["smoke_alarms_installed_num"] == 0) &
    (df["fire_escape_plans_num"] == 0) &
    (df["additional_hazards_num"] == 0)
)
keep_zero = np.random.rand(mask_all_zero.sum()) < KEEP_ZERO_EVENT_FRAC
df = pd.concat([df[~mask_all_zero], df[mask_all_zero].iloc[np.where(keep_zero)[0]]], ignore_index=True)

# Shuffle and add helper columns
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
dt = pd.to_datetime(df["date"])
df["year"] = dt.dt.year
df["month"] = dt.dt.month
df["dow"] = dt.dt.dayofweek
df["is_weekend"] = (df["dow"] >= 5).astype(int)

# Save clean base
CLEAN_CSV = r"C:\Users\syksh\Documents\community_safety_forecasting\data\synthetic_sound_the_alarm_2015_2025.csv"
df.to_csv(CLEAN_CSV, index=False)
CLEAN_CSV, df.shape


('C:\\Users\\syksh\\Documents\\community_safety_forecasting\\data\\synthetic_sound_the_alarm_2015_2025.csv',
 (25931, 11))

## Optional: Inject Gentle Dirty Data (nulls & typos)

In [8]:

df_dirty = df.copy()

if INJECT_DIRTY:
    rng = np.random.default_rng(42)
    # ~3% NaNs in fire_escape_plans_num
    n_null = int(len(df_dirty) * NULL_FRAC_FIRE_ESCAPE)
    null_idx = rng.choice(df_dirty.index, size=n_null, replace=False)
    df_dirty.loc[null_idx, "fire_escape_plans_num"] = np.nan

    # ~2% territory typos (space removal or suffix)
    n_typo = int(len(df_dirty) * TYPO_FRAC_TERRITORY)
    typo_idx = rng.choice(df_dirty.index, size=n_typo, replace=False)
    def tweak_territory(x):
        return x.replace(" ", "") if (" " in x and rng.random() < 0.6) else x + "-X"
    df_dirty.loc[typo_idx, "territory"] = df_dirty.loc[typo_idx, "territory"].apply(tweak_territory)

    # ~0.3% big alarm outliers (cap later in cleaning)
    n_out = int(len(df_dirty) * OUTLIER_FRAC_ALARMS)
    out_idx = rng.choice(df_dirty.index, size=max(n_out,1), replace=False)
    df_dirty.loc[out_idx, "smoke_alarms_installed_num"] = df_dirty.loc[out_idx, "smoke_alarms_installed_num"].clip(lower=40)

DIRTY_CSV = r"C:\Users\syksh\Documents\community_safety_forecasting\data\synthetic_sound_the_alarm_2015_2025_dirty.csv"
df_dirty.to_csv(DIRTY_CSV, index=False)
DIRTY_CSV, df_dirty.shape, df.shape


('C:\\Users\\syksh\\Documents\\community_safety_forecasting\\data\\synthetic_sound_the_alarm_2015_2025_dirty.csv',
 (25931, 11),
 (25931, 11))

## Quick QA Checks

In [9]:

summary = df.describe().T[["mean","std","min","max"]]
dirty_na = df_dirty["fire_escape_plans_num"].isna().mean()
typo_sample = df_dirty[df_dirty["territory"].str.contains("-", na=False)].head(3)

print("CLEAN shape:", df.shape)
print("DIRTY shape:", df_dirty.shape)
print("\nCLEAN summary (first 6 rows):\n", summary.head(6))
print("\nDirty null rate (fire_escape_plans_num):", round(dirty_na,4))
print("\nTerritory typo examples:")
display(typo_sample[["territory"]].head(3))


CLEAN shape: (25931, 11)
DIRTY shape: (25931, 11)

CLEAN summary (first 6 rows):
                                    mean       std     min     max
people_served_num              4.293741  4.459742     0.0    42.0
smoke_alarms_installed_num     3.552775  3.445160     0.0    38.0
fire_escape_plans_num          0.644248  0.794970     0.0     6.0
additional_hazards_num         0.439435  0.661300     0.0     4.0
year                        2020.104932  3.214743  2015.0  2025.0
month                          6.279897  3.344084     1.0    12.0

Dirty null rate (fire_escape_plans_num): 0.03

Territory typo examples:


,territory
111,Inland Empire-X
116,Central Coast-X
121,Inland Empire-X
